# 🌿 03 Testing — Sistem LSTM Bonsai

| Item       | Detail                                      |
|------------|---------------------------------------------|
| **File**   | `notebooks/03_testing.ipynb`                 |
| **Tujuan** | Muat model, jalankan prediksi pada data testing, simpan hasil. |
| **Input**  | `artifacts/model_bonsai_lstm.keras`, `artifacts/data_test.npz`, `artifacts/scaler_bonsai.pkl`, `artifacts/label_info.json` |
| **Output** | `artifacts/predictions.csv`, `artifacts/y_test_labels.csv` |
| **Urutan** | Jalankan setelah: `02_training.ipynb` |

In [1]:
# ── VERIFIKASI LINGKUNGAN ──────────────────────────────────────────────
import sys, os

assert ".venv" in sys.executable or "venv" in sys.executable, (
    "⛔ Kernel bukan dari .venv!\n"
    "Ganti kernel ke: Python (bonsai-lstm)\n"
    f"Kernel saat ini: {sys.executable}"
)
print("✅ Kernel  :", sys.executable)
print("✅ Python  :", sys.version)
# ──────────────────────────────────────────────────────────────────────

✅ Kernel  : D:\bonsai-lstm\.venv\Scripts\python.exe
✅ Python  : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
# ── IMPORT LIBRARY, KONSTANTA, SEED ───────────────────────────────────
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
import random
import warnings
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
print(f"[SEED] Global random seed = {SEED}")

ARTIFACTS_DIR = "../artifacts"
SOIL_MIN      = 0.0
SOIL_MAX      = 100.0

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print("✅ Konfigurasi testing siap.")

[SEED] Global random seed = 42
✅ Konfigurasi testing siap.


In [3]:
# ── RULE-NB-06: Validasi Artefak ───────────────────────────────────────
import json, joblib

REQUIRED = [
    f"{ARTIFACTS_DIR}/model_bonsai_lstm.keras",
    f"{ARTIFACTS_DIR}/data_test.npz",
    f"{ARTIFACTS_DIR}/scaler_bonsai.pkl",
    f"{ARTIFACTS_DIR}/label_info.json",
]
missing = [f for f in REQUIRED if not os.path.exists(f)]
assert not missing, f"⛔ Artefak tidak ada: {missing}"
print("✅ Semua artefak tersedia.")

✅ Semua artefak tersedia.


In [4]:
# ── RULE-TEST-02: Muat Semua Artefak ───────────────────────────────────
test_data  = np.load(f"{ARTIFACTS_DIR}/data_test.npz")
X_test     = test_data["X_test"]
y_test_cls = test_data["y_test_cls"]
y_test_reg = test_data["y_test_reg"]

model  = tf.keras.models.load_model(f"{ARTIFACTS_DIR}/model_bonsai_lstm.keras", safe_mode=False)
scaler = joblib.load(f"{ARTIFACTS_DIR}/scaler_bonsai.pkl")

with open(f"{ARTIFACTS_DIR}/label_info.json") as f:
    label_info = json.load(f)

print(f"[LOAD] X_test shape : {X_test.shape}")
print(f"[LOAD] Model        : {model.name}")


[LOAD] X_test shape : (2011, 24, 3)


[LOAD] Model        : bonsai_lstm_residual


In [5]:
# ── RULE-TEST-03: Menjalankan Prediksi ─────────────────────────────────────────
y_pred = model.predict(X_test, verbose=0)
y_model_prob = y_pred[0].flatten()      # Output klasifikasi model (diagnostik)
y_pred_soil_raw = y_pred[1].flatten()  # Output regresi (scaled 0-1)

# Kembalikan ke skala % (kalikan 100)
y_pred_soil = np.clip(y_pred_soil_raw * 100.0, SOIL_MIN, SOIL_MAX)

soil_threshold = float(label_info.get("soil_threshold", 60.0))
y_pred_class = (y_pred_soil < soil_threshold).astype(int)

# Probabilitas operasional dibuat konsisten dengan rule agronomi:
# semakin rendah prediksi kelembapan tanah, semakin tinggi peluang perlu disiram.
y_pred_prob = np.clip(1.0 - (y_pred_soil / 100.0), 0.0, 1.0)

print(f"[PREDICT] Total prediksi          : {len(y_pred_prob)}")
print(f"[PREDICT] Prediksi Siram (1)      : {y_pred_class.sum()}")
print(f"[PREDICT] Prediksi Tidak (0)      : {(y_pred_class==0).sum()}")
print(f"[PREDICT] Prob. operasional rata2 : {y_pred_prob.mean():.4f}")
print(f"[PREDICT] Prob. model rata2       : {y_model_prob.mean():.4f}")


[PREDICT] Total prediksi          : 2011
[PREDICT] Prediksi Siram (1)      : 628
[PREDICT] Prediksi Tidak (0)      : 1383
[PREDICT] Prob. operasional rata2 : 0.3713
[PREDICT] Prob. model rata2       : 0.3273


In [6]:
# ── RULE-TEST-04: Simpan Hasil Prediksi ────────────────────────────────
pred_df = pd.DataFrame({
    "y_pred_prob"    : y_pred_prob,
    "y_pred_class"   : y_pred_class,
    "y_pred_soil_pct": y_pred_soil,
})
pred_df.to_csv(f"{ARTIFACTS_DIR}/predictions.csv", index=False)

actual_df = pd.DataFrame({
    "y_true_class"   : y_test_cls,
    "y_true_soil_pct": y_test_reg,
})
actual_df.to_csv(f"{ARTIFACTS_DIR}/y_test_labels.csv", index=False)

print("[SAVE] ✅ artifacts/predictions.csv")
print("[SAVE] ✅ artifacts/y_test_labels.csv")

[SAVE] ✅ artifacts/predictions.csv
[SAVE] ✅ artifacts/y_test_labels.csv


In [7]:
# ── RULE-TEST-05: Sanity Check Hasil Prediksi ───────────────────────────
check_df = pd.concat([actual_df, pred_df], axis=1)
print("\n[SANITY CHECK] 10 sampel pertama:")
print(check_df.head(10).to_string(index=False))

correct = (check_df["y_true_class"] == check_df["y_pred_class"]).sum()
total   = len(check_df)
print(f"\n[SANITY] Prediksi benar: {correct}/{total} = {correct/total*100:.2f}%")


[SANITY CHECK] 10 sampel pertama:
 y_true_class  y_true_soil_pct  y_pred_prob  y_pred_class  y_pred_soil_pct
            0            64.50     0.281444             0        71.855576
            0            64.11     0.362694             0        63.730568
            0            69.61     0.366542             0        63.345848
            0            70.27     0.310857             0        68.914330
            0            76.00     0.304178             0        69.582191
            0            71.40     0.246194             0        75.380562
            0            68.90     0.292712             0        70.728783
            0            79.77     0.317848             0        68.215218
            0            69.43     0.207944             0        79.205605
            0            78.10     0.312300             0        68.769989

[SANITY] Prediksi benar: 1921/2011 = 95.52%


## 📊 Ringkasan Testing

**File yang dihasilkan:**
- `predictions.csv` → Probabilitas, kelas prediksi, estimasi soil %
- `y_test_labels.csv` → Label aktual & nilai aktual soil %

**Langkah selanjutnya:**
Jalankan `notebooks/04_evaluasi.ipynb` untuk menghitung metrik dan visualisasi.